In [ ]:
!pip uninstall torch -y
!pip install torch torchvision transformers faiss-cpu pillow requests
!pip install fashion-clip
!pip install tqdm


import os
import json
import torch
import random
import numpy as np
from PIL import Image
from io import BytesIO
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from torch.optim import AdamW
import torch.nn.functional as F
import requests

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [4]:
from torch.utils.data import Dataset
from PIL import Image
import requests
from io import BytesIO

class FashionDataset(Dataset):
    def __init__(self, json_file):
        # Load JSON
        import json
        with open(json_file, "r", encoding="utf-8") as f:
            self.items = json.load(f)  # <-- make sure this line exists
        print(f"Loaded {len(self.items)} items")  # optional debug

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]

        try:
            # If item is a dict
            if isinstance(item, dict):
                img_path_or_url = item.get("image", "")
                text = item.get("caption", "fashion item")
            else:  # assume item is just a string URL
                img_path_or_url = item
                text = "fashion item"

            # Load image
            if img_path_or_url.startswith("http"):
                img = Image.open(BytesIO(requests.get(img_path_or_url).content)).convert("RGB")
            else:
                img = Image.open(img_path_or_url).convert("RGB")

            return img, text, img_path_or_url
        except:
            return None  # skip broken items

In [9]:
def collate_skip_none(batch):
    """Filter out None samples; do not unpack anything here."""
    return [x for x in batch if x is not None]

In [10]:
def clip_loss(image_embeds, text_embeds):
    # Normalize embeddings (already done in your loop)
    logits = image_embeds @ text_embeds.T  # cosine similarity
    labels = torch.arange(len(image_embeds)).to(image_embeds.device)
    
    loss_i2t = torch.nn.CrossEntropyLoss()(logits, labels)
    loss_t2i = torch.nn.CrossEntropyLoss()(logits.T, labels)
    return (loss_i2t + loss_t2i) / 2


In [11]:
class EarlyStopping:
    def __init__(self, patience=3):
        self.patience = patience
        self.best_loss = float("inf")
        self.counter = 0

    def check(self, val_loss, model, save_path):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            os.path.join(save_path, "best_model.pt")
            print(f" Saved best model → {os.path.join(save_path,'best_model.pt')}")
        else:
            self.counter += 1
        return self.counter >= self.patience

In [ ]:
def train_pipeline(json_file,
                   model_name="patrickjohncyh/fashion-clip",
                   batch_size=16,
                   epochs=1,
                   save_root="fashionclip_finetuned_6images_peritem"):

    import os
    from torch.utils.data import DataLoader
    from transformers import CLIPProcessor, CLIPModel
    from torch.optim import AdamW
    from sklearn.model_selection import train_test_split
    from tqdm import tqdm

    os.makedirs(save_root, exist_ok=True)

    # ========= LOAD DATA =========
    with open(json_file, "r", encoding="utf-8") as f:
        data_list = json.load(f)

    train_data, test_data = train_test_split(data_list, test_size=0.1, random_state=42)
    train_data, val_data = train_test_split(train_data, test_size=0.1, random_state=42)

    train_ds = FashionDataset(train_data)
    val_ds = FashionDataset(val_data)
    test_ds = FashionDataset(test_data)

    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_skip_none)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_skip_none)
    test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_skip_none)

    # ========= LOAD MODEL =========
    processor = CLIPProcessor.from_pretrained(model_name, use_fast=True)
    model = CLIPModel.from_pretrained(model_name).to(device)

    # Freeze & unfreeze layers for fine-tuning
    for p in model.parameters(): p.requires_grad = False
    for layer in model.vision_model.encoder.layers[-2:]:
        for p in layer.parameters(): p.requires_grad = True
    for layer in model.text_model.encoder.layers[-2:]:
        for p in layer.parameters(): p.requires_grad = True
    for p in model.visual_projection.parameters(): p.requires_grad = True
    for p in model.text_projection.parameters(): p.requires_grad = True

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-6, weight_decay=1e-4)

    early = EarlyStopping(patience=3)

    # ========= TRAIN LOOP =========
    for epoch in range(epochs):
        print(f"\n===== Epoch {epoch+1}/{epochs} =====")
        total_train_loss = 0
        model.train()

        for batch in tqdm(train_dl):
            if batch is None:
                continue

            images, texts = batch
            inputs = processor(images=images, text=texts,
                               return_tensors="pt", padding=True).to(device)

            # Forward pass & embeddings
            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                pixel_values=inputs["pixel_values"],
                return_dict=True
            )
            img_emb = outputs.image_embeds / outputs.image_embeds.norm(dim=1, keepdim=True)
            txt_emb = outputs.text_embeds / outputs.text_embeds.norm(dim=1, keepdim=True)

            # Contrastive loss
            loss = clip_loss(img_emb, txt_emb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()

        avg_train_loss = total_train_loss / len(train_dl)
        print(f"Epoch {epoch+1} Train Loss: {avg_train_loss:.4f}")

        # ========= VALIDATION =========
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_dl:
                if batch is None: continue
                images, texts = batch
                inputs = processor(images=images, text=texts,
                                   return_tensors="pt", padding=True).to(device)

                outputs = model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    pixel_values=inputs["pixel_values"],
                    return_dict=True
                )
                img_emb = outputs.image_embeds / outputs.image_embeds.norm(dim=1, keepdim=True)
                txt_emb = outputs.text_embeds / outputs.text_embeds.norm(dim=1, keepdim=True)
                val_loss += clip_loss(img_emb, txt_emb).item()

        avg_val_loss = val_loss / len(val_dl)
        print(f"Epoch {epoch+1} Val Loss: {avg_val_loss:.4f}")

        # Early stopping check
        if early.check(avg_val_loss, model, save_root):
            print("⛔ Early stopping triggered.")
            break

    # ========= TESTING =========
    model.eval()
    test_loss = 0
    with torch.no_grad():
        for batch in test_dl:
            if batch is None: continue
            images, texts = batch
            inputs = processor(images=images, text=texts,
                               return_tensors="pt", padding=True).to(device)

            outputs = model(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                pixel_values=inputs["pixel_values"],
                return_dict=True
            )
            img_emb = outputs.image_embeds / outputs.image_embeds.norm(dim=1, keepdim=True)
            txt_emb = outputs.text_embeds / outputs.text_embeds.norm(dim=1, keepdim=True)
            test_loss += clip_loss(img_emb, txt_emb).item()

    avg_test_loss = test_loss / len(test_dl)
    print(f"🔥 Test Loss: {avg_test_loss:.4f}")

    # ========= SAVE FINAL MODEL + PROCESSOR (BEST PRACTICE) =========
    model.save_pretrained(save_root, safe_serialization=True)  # saves weights as .safetensors
    processor.save_pretrained(save_root)                        # saves tokenizer/processor
    print(f"✅ Final fine-tuned model saved to: {save_root}")

    return model, processor


In [ ]:
model, processor = train_pipeline( "/kaggle/input/datasets/nadinemohsen804/first6-images/products_first6_images.json", epochs=1, batch_size=16 )

In [ ]:
import shutil

shutil.make_archive(
    '/kaggle/working/fashionclip_finetuned_6images_peritem',
    'zip',
    '/kaggle/working/fashionclip_finetuned_6images_peritem'
)

In [12]:
!pip install torch torchvision transformers faiss-cpu pillow requests
!pip install fashion-clip
!pip install tqdm

import json
import torch
import numpy as np
import faiss
import requests
import os

from torch.utils.data import Dataset, DataLoader
from PIL import Image
from io import BytesIO
from transformers import CLIPProcessor, CLIPModel
from tqdm import tqdm

def append_numpy(file_path, array):
    """Append rows to a .npy file (safe incremental save)."""
    if not os.path.exists(file_path):
        np.save(file_path, array)
    else:
        old = np.load(file_path)
        combined = np.concatenate([old, array], axis=0)
        np.save(file_path, combined)


def append_jsonl(file_path, data_list):
    """Append list of strings to a JSONL file."""
    with open(file_path, "a", encoding="utf-8") as f:
        for item in data_list:
            f.write(json.dumps(item) + "\n")




# ----------------------------------------------------
# LOAD CLIP
# ----------------------------------------------------
clip_processor = CLIPProcessor.from_pretrained("/kaggle/input/models/nadinemohsen804/trial2/pytorch/default/1")
clip_model = CLIPModel.from_pretrained("/kaggle/input/models/nadinemohsen804/trial2/pytorch/default/1").to(device)

clip_model.eval()



# ----------------------------------------------------
# FILES FOR INCREMENTAL SAVE
# ----------------------------------------------------
IMG_FILE = "image_embeddings.npy"
TXT_FILE = "text_embeddings.npy"
URL_FILE = "image_urls.jsonl"

# Remove old files if they exist (optional)
for f in [IMG_FILE, TXT_FILE, URL_FILE]:
    if os.path.exists(f):
        os.remove(f)
        print("Deleted old file:", f)


# ----------------------------------------------------
# GENERATE & SAVE EMBEDDINGS INCREMENTALLY
# ----------------------------------------------------
print("\n🔄 Generating + Saving embeddings batch-by-batch...\n")
json_file = "/kaggle/input/datasets/nadinemohsen804/newdata/dataset(28k).json"  # replace with your dataset path
dataset = FashionDataset(json_file)

loader = DataLoader(dataset,batch_size=16,shuffle=False,num_workers=2,collate_fn=collate_skip_none)  # skip None items)
with torch.no_grad():
    for batch in tqdm(loader, desc="Embedding"):
        if batch is None:
            continue

        images, texts, urls = map(list, zip(*batch))  # convert each to list

        # Process batch with CLIP
        inputs = clip_processor(
            images=images,
            text=texts,
            return_tensors="pt",
            padding=True
        ).to(device)

        img_embeds = clip_model.get_image_features(inputs["pixel_values"])
        txt_embeds = clip_model.get_text_features(
            inputs["input_ids"],
            inputs["attention_mask"]
        )

        # Normalize
        img_embeds = img_embeds / img_embeds.norm(dim=1, keepdim=True)
        txt_embeds = txt_embeds / txt_embeds.norm(dim=1, keepdim=True)

        img_np = img_embeds.cpu().numpy()
        txt_np = txt_embeds.cpu().numpy()

        # -------- SAVE INCREMENTALLY --------
        append_numpy(IMG_FILE, img_np)
        append_numpy(TXT_FILE, txt_np)
        append_jsonl(URL_FILE, urls)


print("\n✅ Done! All embeddings saved safely.")
print("📁 image_embeddings.npy")
print("📁 text_embeddings.npy")
print("📁 image_urls.jsonl")



🔄 Generating + Saving embeddings batch-by-batch...

Loaded 28136 items


Embedding:  18%|█▊        | 321/1759 [17:25<57:16,  2.39s/it]  /usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Embedding:  18%|█▊        | 325/1759 [17:40<52:48,  2.21s/it]  /usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Embedding: 100%|██████████| 1759/1759 [1:53:25<00:00,  3.87s/it]  


✅ Done! All embeddings saved safely.
📁 image_embeddings.npy
📁 text_embeddings.npy
📁 image_urls.jsonl


build faiss


In [13]:
import numpy as np

# Load embeddings and URLs
image_embeddings = np.load("image_embeddings.npy")   # shape: (N, dim)
text_embeddings = np.load("text_embeddings.npy")     # if needed
with open("image_urls.jsonl", "r", encoding="utf-8") as f:
    all_image_urls = [line.strip() for line in f]

# Verify shapes
print("Image embeddings:", image_embeddings.shape)
print("Text embeddings:", text_embeddings.shape if 'text_embeddings' in locals() else "N/A")
print("URLs count:", len(all_image_urls))


Image embeddings: (28026, 512)
Text embeddings: (28026, 512)
URLs count: 28026


In [14]:
embedding_dim = image_embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)  # cosine similarity (normalized vectors)
index.add(image_embeddings)

print(f"FAISS index built with {index.ntotal} vectors.")


FAISS index built with 28026 vectors.


# Safe FAISS search function by query

In [24]:
query =  "yellow cardigan"
results = search_faiss(query, top_k=5)

for r in results:
    print(r)


"https://cdn.shopify.com/s/files/1/0644/7615/2055/files/7V5A4557_f971073b-d41e-498d-a28c-1dfdb181b377.jpg"
"https://pernov-a.com/cdn/shop/files/IMG_5119.webp"
"https://bucket.zammit.shop/active-storage/4g2eyoxqdobqb4pwyvomqtbhnuz1"
"https://hkdesigns-eg.com/cdn/shop/files/Photo21-06-2023_82841PM.jpg"
"https://hkdesigns-eg.com/cdn/shop/products/Photo28-03-2022_54806PM.jpg"


In [15]:
import torch
from transformers import CLIPProcessor, CLIPModel

def search_faiss(query_text, top_k=5):
    # Encode query
    inputs = clip_processor(text=[query_text], return_tensors="pt").to(device)
    with torch.no_grad():
        q = clip_model.get_text_features(inputs["input_ids"], inputs["attention_mask"])
        q = q / q.norm(p=2, dim=-1, keepdim=True)

    q = q.cpu().numpy().astype('float32')

    # Search
    D, I = index.search(q, top_k)

    # Return safe results
    results = []
    for idx in I[0]:
        if 0 <= idx < len(all_image_urls):
            results.append(all_image_urls[idx])
    return results



def search_image_to_image(query_image_pil, top_k=5):
    clip_model.eval()

    # Convert image to embedding
    inputs = clip_processor(
        images=[query_image_pil],
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        img_emb = clip_model.get_image_features(inputs["pixel_values"])
        img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
        img_emb = img_emb.cpu().numpy()

    # FAISS search
    D, I = index.search(img_emb, top_k)

    results = [all_image_urls[i] for i in I[0]]
    return results


In [28]:
query_img = Image.open("/kaggle/input/datasets/nadinemohsen804/image5/6303DB83-65CF-478E-8956-E05758945F34.webp").convert("RGB")
result_urls = search_image_to_image(query_img, top_k=5)

for url in result_urls:
    print(url)


"https://cdn.shopify.com/s/files/1/0097/4746/4251/files/8894A8F9-2705-426B-849A-30A1B21C2512.jpg"
"https://cdn.shopify.com/s/files/1/0097/4746/4251/files/8DAC52E0-4848-481E-A586-75648ED7B313.jpg"
"https://cdn.shopify.com/s/files/1/0097/4746/4251/files/F8BF107B-6197-4E8A-8137-AF7353276C5C.jpg"
"https://cdn.shopify.com/s/files/1/0900/6045/6256/files/IMG_1550-scaled.jpg"
"https://cdn.shopify.com/s/files/1/0759/7192/0186/files/9DB25665-0214-4A00-BFB4-6CD349D72604.jpg"


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import requests
from io import BytesIO
from tqdm import tqdm
from transformers import CLIPProcessor, CLIPModel
import json
from sklearn.model_selection import train_test_split



# -------------------------------
# 1) Evaluation Dataset
# -------------------------------


# -------------------------------
# 2) Collate function

# 3) Compute embeddings
# -------------------------------
def compute_embeddings(model, processor, dataloader, device):
    all_img_emb = []
    all_txt_emb = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Embedding Eval Set"):
            if batch is None:
                continue

            images, texts = batch

            batch_inputs = processor(
                images=images,
                text=texts,
                return_tensors="pt",
                padding=True
            ).to(device)

            outputs = model(
                input_ids=batch_inputs["input_ids"],
                attention_mask=batch_inputs["attention_mask"],
                pixel_values=batch_inputs["pixel_values"],
                return_dict=True
            )

            img_emb = outputs.image_embeds.cpu()
            txt_emb = outputs.text_embeds.cpu()

            all_img_emb.append(img_emb)
            all_txt_emb.append(txt_emb)

    return torch.cat(all_img_emb), torch.cat(all_txt_emb)

# -------------------------------
# 4) Recall@K metric
# -------------------------------
def recall_at_k(similarity_matrix, k=1):
    correct = 0
    for i in range(similarity_matrix.shape[0]):
        top_k = similarity_matrix[i].argsort(descending=True)[:k]
        if i in top_k:
            correct += 1
    return correct / similarity_matrix.shape[0]


#3) Semantic Recall@K (Image→Image)
# -------------------------------
def semantic_recall_at_k(image_embeds, k=10, threshold=0.75):

    image_embeds = F.normalize(image_embeds, dim=-1)
    similarity = image_embeds @ image_embeds.T

    N = similarity.size(0)
    total_score = 0.0

    for i in range(N):

        sim_scores = similarity[i].clone()
        sim_scores[i] = -1e9  # remove self similarity

        topk_indices = torch.topk(sim_scores, k=k).indices
        topk_sim_values = similarity[i, topk_indices]

        semantic_hits = (topk_sim_values > threshold).float().sum()
        total_score += (semantic_hits / k).item()

    return total_score / N


# -------------------------------
# 5) Full Evaluation Function
# -------------------------------
def evaluate_retrieval(test_dataset, model, processor, batch_size=16):

    loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=collate_skip_none
    )

    print("Computing embeddings...")
    img_emb, txt_emb = compute_embeddings(model, processor, loader, device)

    # Normalize
    img_emb = F.normalize(img_emb, dim=-1)
    txt_emb = F.normalize(txt_emb, dim=-1)

    # =============================
    # Image ↔ Text
    # =============================
    sim_i2t = img_emb @ txt_emb.T
    sim_t2i = txt_emb @ img_emb.T

    # =============================
    # Image ↔ Image
    # =============================
    sim_i2i = img_emb @ img_emb.T
    sim_i2i.fill_diagonal_(-1e9)

    # =============================
    # Exact Recall Metrics
    # =============================
    results = {
        "Image→Text R@1":  recall_at_k(sim_i2t, 1),
        "Image→Text R@5":  recall_at_k(sim_i2t, 5),
        "Image→Text R@10": recall_at_k(sim_i2t, 10),

        "Text→Image R@1":  recall_at_k(sim_t2i, 1),
        "Text→Image R@5":  recall_at_k(sim_t2i, 5),
        "Text→Image R@10": recall_at_k(sim_t2i, 10),
    }

    # =============================
    # Semantic Metric
    # =============================
    semantic_r10 = semantic_recall_at_k(
        img_emb,
        k=10,
        threshold=0.75
    )

    results["Semantic Recall@10 (0.75)"] = semantic_r10

    # Print results
    print("\n===== Exact Recall =====")
    for k, v in results.items():
        if isinstance(v, float):
            print(f"{k}: {v*100:.2f}%")

    return results



# -------------------------------
# 6) Load dataset splits
# -------------------------------
with open("/kaggle/input/datasets/nadinemohsen804/newdata/dataset(28k).json", "r", encoding="utf-8") as f:
    data_list = json.load(f)

train_data, test_data = train_test_split(data_list, test_size=0.1, random_state=42)
train_data, val_data = train_test_split(train_data, test_size=0.1, random_state=42)

# Only test split for evaluation
eval_ds = FashionDataset(test_data)

# -------------------------------
# 7) Load fine-tuned model
# -------------------------------
model_path = "/kaggle/input/models/nadinemohsen804/trial2/pytorch/default/1"
processor = CLIPProcessor.from_pretrained(model_path)
model = CLIPModel.from_pretrained(model_path).to(device)
model.eval()

# -------------------------------
# 8) Run evaluation
# -------------------------------
results = evaluate_retrieval(eval_ds, model, processor, batch_size=16)
print(results)
print("Evaluation done!")
print("Top-5 indices shape:", results["topk_indices"].shape) 

In [ ]:
# After evaluation
img_emb, txt_emb = compute_embeddings(model, processor, DataLoader(
    eval_ds, batch_size=16, shuffle=False, collate_fn=collate_skip_none
), device)

# Normalize
img_emb = torch.nn.functional.normalize(img_emb, dim=-1)
txt_emb = torch.nn.functional.normalize(txt_emb, dim=-1)

# Convert to numpy for easier search
img_emb_np = img_emb.cpu().numpy()
txt_emb_np = txt_emb.cpu().numpy()

In [ ]:
import numpy as np
import torch.nn.functional as F

def search_text_to_image(query_text, model, processor, img_emb, all_image_urls, top_k=5):
    # Encode query text
    inputs = processor(text=[query_text], return_tensors="pt").to(device)

    with torch.no_grad():
        q = model.get_text_features(
            inputs["input_ids"],
            inputs["attention_mask"]
        )
        q = F.normalize(q, dim=-1)

    q = q.cpu().numpy()  # (1, d)

    # Cosine similarity
    sims = (img_emb @ q.T).squeeze()  # (N,)

    # Top-K
    topk_idx = np.argsort(-sims)[:top_k]

    return [all_image_urls[i] for i in topk_idx]

model_path = "/kaggle/input/models/nadinemohsen804/trial2/pytorch/default/1"
processor = CLIPProcessor.from_pretrained(model_path)
model = CLIPModel.from_pretrained(model_path).to(device)
results = search_text_to_image("yellow summer dress", model, processor, img_emb_np, all_image_urls)
for r in results:
    print(r)

In [ ]:
def search_image_to_image_no_faiss(query_image_pil, model, processor, img_emb, all_image_urls, top_k=5):
    # Encode image
    inputs = processor(images=[query_image_pil], return_tensors="pt").to(device)
    with torch.no_grad():
        q = model.get_image_features(inputs["pixel_values"])
        q = F.normalize(q, dim=-1)

    q = q.cpu().numpy()  # (1, d)

    # Cosine similarity
    sims = (img_emb @ q.T).squeeze()  # (N,)

    topk_idx = np.argsort(-sims)[:top_k]

    return [all_image_urls[i] for i in topk_idx]

model_path = "/kaggle/input/models/nadinemohsen804/trial2/pytorch/default/1"
processor = CLIPProcessor.from_pretrained(model_path)
model = CLIPModel.from_pretrained(model_path).to(device)
query_img = Image.open("/kaggle/input/datasets/nadinemohsen804/img-test/Screenshot 2026-02-06 160642.png").convert("RGB")
results = search_image_to_image_no_faiss(query_img, model, processor, img_emb_np, all_image_urls)

for r in results:
    print(r)